In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel, StableDiffusionXLPipeline
from diffusers.utils import load_image
from PIL import Image, ImageOps

In [ ]:
controlnet = ControlNetModel.from_pretrained(
    "ShermanG/ControlNet-Standard-Lineart-for-SDXL",
    torch_dtype = torch.float16
)

In [ ]:
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "cagliostrolab/animagine-xl-3.1",
    controlnet = controlnet,
    torch_dtype = torch.float16
).to("cuda")

In [ ]:
base_pipe = StableDiffusionXLPipeline.from_pretrained(
    "cagliostrolab/animagine-xl-3.1",
    torch_dtype = torch.float16
).to("cuda")

In [ ]:
#メモリ節約らしい。。
pipe.enable_model_cpu_offload()

In [ ]:
#テスト
image = base_pipe(
    prompt="anime girl, colorful illustration",
    num_inference_steps=20
).images[0]

image.save("test.png")


In [ ]:
lineart = Image.open("/content/drive/MyDrive/深層学習/difusser/lineart.png").convert("RGB")

In [ ]:
# 白背景黒線なら反転（重要）
lineart = ImageOps.invert(lineart)

In [ ]:
lineart

In [ ]:
lineart = lineart.resize((1024, 1024))

In [ ]:


# --- 生成 ---
image = pipe(
    prompt="anime style coloring, vibrant colors, clean shading",
    negative_prompt="blurry, low quality, messy",
    image=lineart,
    controlnet_conditioning_scale=1.0,
    num_inference_steps=30,
    width=512,
    height=512
).images[0]

image.save("/content/output.png")